# Analysis of samples from TNP-KR

## TODO

$$
\def\ctx{\mathrm{ctx}}
\def\test{\mathrm{test}}
\def\autoreg{\mathrm{autoreg}}
$$

- plot difference heatmaps
- plot monte carlo traces (do things even converge?)
- forward error? i.e. distribution of $q_\theta^\autoreg(\test | \ctx)p(\ctx)$ vs $p(\ctx, \test)$ vs $q_\theta(\test | \ctx)p(\ctx)$


## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from omegaconf import OmegaConf
from sps.kernels import rbf

plt.style.use("default")
results_dir = Path(
    "/Users/pgrynfelder/Library/CloudStorage/GoogleDrive-wadh6460@ox.ac.uk/My Drive/results"
)


runs = sorted(
    list(set(map(lambda x: x.parent, results_dir.glob("*/*.npy")))),
    key=lambda path: path.stat().st_mtime,
)

In [ ]:
# load model

prompt = "Which run? Type a number.\n" + "\n".join(
    [f"{i}: {run.name}" for i, run in enumerate(runs)]
)
print(prompt)
i = int(input(prompt))
run = runs[i]
print(f"Selected {run.name}")

# Load data
data = dict()
for file in run.glob("*.npy"):
    data[file.stem] = np.load(file)

# Simplify dimensionality
for key in data.keys():
    if (
        key.startswith("paths_")
        or key.startswith("s_")
        or key.startswith("diagonal_")
        or key.startswith("f_")
    ):
        data[key] = data[key].squeeze(-1)
data["s"] = data["s"].squeeze(-1)
data["f"] = data["f"].squeeze(-1)


# Sort s_test
idx = np.argsort(data["s_test"])
data["s_test"] = data["s_test"][idx]
for key in data.keys():
    if key.startswith("paths_"):
        data[key] = data[key][:, idx]
data["diagonal_mu"] = data["diagonal_mu"][idx]
data["diagonal_var"] = data["diagonal_var"][idx]

print("Seed:", data["seed"])
for key, value in sorted(data.items()):
    print(key, value.shape)

if (run / "config.yaml").exists():
    config = OmegaConf.load(run / "config.yaml")
else:
    config = None

paths = sorted([key for key in data.keys() if "paths_" in key])
if "paths_diagonal" in paths:
    paths.remove("paths_diagonal")  # no need to see this for most plots
strategies = [key.lstrip("paths_") for key in paths]

summary = pd.DataFrame()

In [ ]:
# Print config
from json import dumps

print(dumps(OmegaConf.to_object(config), indent=4))
print("Variance", data["var"])
print("Lengthscale", data["ls"])


In [ ]:
# Utils
from jax.experimental import enable_x64


def grid(num_items: int, ncols: int = 3, **kwargs):
    """`plt.subplots` with better defaults"""
    nrows = (num_items - 1) // ncols + 1
    figsize = (ncols * 5, nrows * 5)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kwargs)
    fig.set_constrained_layout(True)
    for ax in axes.flat[num_items:]:
        ax.axis("off")
    return fig, axes


def logpdf(s, f, kernel, var, ls):
    s = s.astype(np.float64)
    f = f.astype(np.float64)
    with enable_x64():
        cov = kernel(s, s, var, ls)

    return stats.multivariate_normal.logpdf(f, cov=cov)

## Analytic solution

Note that it assumes we know the kernel, variance, lengthscale, and observation noise so requires strictly more information than the model.

In [ ]:
from dl4bi.meta_learning.autoregressive import analytic_gp

analytic_mu, analytic_cov = analytic_gp(
    data["s_ctx"],
    data["f_ctx"],
    data["s_test"],
    kernel=rbf,
    var=data["var"],
    ls=data["ls"],
    noise=config.data.obs_noise,
)
analytic_mu, analytic_cov = np.array(analytic_mu), np.array(analytic_cov)

data["analytic_mu"] = analytic_mu
data["analytic_cov"] = analytic_cov
del analytic_mu, analytic_cov

## Metrics

### L2 distance from analytical solution

The $L^2$ squared distance between continuous distributions is given by

$$ \int |p(x) - q(x)|^2 \mathrm{d} x .$$

As we have samples from $q$ available along with 'densities' but can't access the pdf directly, we will use an importance sampling estimate:

$$ \def\E{\mathbb{E}}
\widehat\E_{x \sim q}[|p(x) - q(x)|^2 / q(x)]
$$

Note this is only accurate if both distributions have the same support.

### KL divergence

We can estimate the reverse KL-divergence from the analytic solution
$$ 
\mathrm{KL}\left(q^\autoreg(\cdot | \ctx)\,\|\, p(\cdot | \ctx)\right) = \E_{Y_{s_\test} \sim q^\autoreg(\cdot | \ctx)}\left[\log \frac{q^\autoreg(Y_{s_\test} | \ctx)}{p(Y_{s_\test} | \ctx)}\right]
$$
using the autoregressive samples from the model.

In [ ]:
def KL_estimate(paths, log_q, p_mu, p_cov, jitter=1e-9):
    """KL estimate but using the analytically derived posterior variance and mean"""
    p_dist = stats.multivariate_normal(p_mu, p_cov + jitter * np.eye(p_cov.shape[0]))
    log_p = p_dist.logpdf(paths)
    return np.mean(log_q - log_p)


def H_estimate(log_densities):
    return -np.mean(log_densities)


for path in paths:
    strategy = path.split("_")[-1]
    estimate = KL_estimate(
        data[path],
        data[f"densities_{strategy}"],
        data["analytic_mu"],
        data["analytic_cov"],
        jitter=1e-5,
    )
    summary.loc[path, "KL2"] = estimate
    summary.loc[path, "H(q)"] = H_estimate(data[f"densities_{strategy}"])

print(summary.KL2.sort_values())
print(summary["H(q)"].sort_values())


## Covariance

In [ ]:
images = {
    "analytic": data["analytic_cov"],
    "diagonal": np.diag(data["diagonal_var"]),
}
for key in paths:
    images[key] = np.cov(data[key].T)

vmin = min([image.min() for image in images.values()])
vmax = max([image.max() for image in images.values()])
v = max(abs(vmin), abs(vmax))

plt.tight_layout()
cmap = plt.get_cmap("RdBu")
norm = plt.Normalize(vmin=-v, vmax=v)
im = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

fig, axes = grid(len(images), sharex=True, sharey=True)
for (path, image), ax in zip(images.items(), axes.flat):
    image = np.atleast_2d(image)
    ax.imshow(image, cmap=cmap, norm=norm)

    ctx = [np.searchsorted(data["s_test"], s) for s in data["s_ctx"]]
    ax.hlines(ctx, *ax.get_xlim(), "g", linewidth=0.3)
    ax.vlines(ctx, *ax.get_ylim(), "g", linewidth=0.3)

    ax.set_title(path)
    ax.axis("off")
fig.colorbar(im, ax=axes.flatten())
fig.suptitle(
    "Covariance matrices for different path sampling strategies + analytic solutions"
)
plt.show()


for path, image in images.items():
    summary.loc[path, "Covariance - Frobenius distance from analytic"] = np.linalg.norm(
        image - images["analytic"], ord="fro"
    )

print(summary["Covariance - Frobenius distance from analytic"].sort_values())

## Means + 95% CI

In [ ]:
means = {
    "analytic": data["analytic_mu"],
    "diagonal": data["diagonal_mu"],
} | {path: np.mean(data[path], axis=0) for path in paths}
variances = {
    "analytic": np.diag(data["analytic_cov"]),
    "diagonal": data["diagonal_var"],
} | {path: np.var(data[path], axis=0) for path in paths}


fig, axes = grid(len(means))

for path, ax in zip(means.keys(), axes.flat):
    mu = means[path]
    var = variances[path]
    lo = mu - 2 * np.sqrt(var)
    hi = mu + 2 * np.sqrt(var)
    ax.plot(data["s_test"], mu, ".", alpha=0.5)
    ax.fill_between(data["s_test"], lo, hi, alpha=0.2)
    ax.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context", linewidth=500)
    # ax.plot(data["s_test"], data["analytic_mu"], "--", label="analytic mean", alpha=0.5)
    ax.legend()
    ax.set_title(path)

fig.suptitle("$\mu \pm 2\sigma$")
plt.show()


## Mean

In [ ]:
fig, axes = grid(len(means))
for (path, mean), ax in zip(means.items(), axes.flat):
    ax.set_title(path)
    ax.plot(data["s_test"], mean)
    # ax.plot(data["s_test"], data["analytic_mu"], "--", label="Analytic", alpha=0.5)
    ax.plot(data["s_ctx"], data["f_ctx"], "g+", label="Context")

fig.suptitle("$\mu$")
plt.legend()
plt.plot()

## Variance

In [ ]:
fig, axes = grid(len(variances))
for (path, variance), ax in zip(variances.items(), axes.flat):
    ax.set_title(path)
    ax.plot(data["s_test"], variance)
    # ax.plot(
    #     data["s_test"],
    #     np.diag(data["analytic_cov"]),
    #     "--",
    #     alpha=0.5,
    #     label="Analytic variance",
    # )
    ax.plot(data["s_ctx"], 0 * data["s_ctx"], "g+", label="Context")
fig.set_constrained_layout(True)
fig.suptitle("Variance")
plt.legend()
plt.show()

## Example sample paths

In [ ]:
Np = 4

indices = np.random.choice(data["paths_ltr"].shape[0], (Np,), replace=False)

fig, axes = grid(len(paths) * Np, ncols=Np, sharey=True, sharex=True)
for path, axs in zip(paths, axes):
    axs[0].set_ylabel(path)
    for idx, ax in zip(indices, axs):
        ax.plot(data["s_test"], data[path][idx])
        ax.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context")
fig.suptitle("Autoregressive sample paths")

plt.show()

In [ ]:
plt.title("A path from the true gp")

idx = data["s"].argsort()
plt.plot(data["s"][idx], data["f"][idx], "-")
plt.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context")

## Estimation traces

In [ ]:
path = "paths_ltr"
N_locations = 100
ns = np.arange(11, data[path].shape[0] + 1)  # dropping first 10 steps from the plot

np.random.seed(0)
locations = np.random.choice(data["s_test"].shape[0], N_locations, replace=False)
locations.sort()

In [ ]:
# plot running means

fig, axes = grid(N_locations, ncols=5)
for location, ax in zip(locations, axes.flat):
    running_sum = data[path][:, location].cumsum()[ns - 1]
    running_mean = running_sum / ns
    ax.plot(ns, running_mean, label="Running mean", linewidth=2)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"Running means for {path} at {N_locations} randomly chosen locations")
plt.show()


### plot running variances

In [ ]:
ns = np.arange(
    11, data[path].shape[0] + 1
)  # note dropping the first 10 steps from the plots
running_variances = np.stack([np.var(data[path][:n, locations], axis=0) for n in ns]).T

fig, axes = grid(N_locations, ncols=5)
for location, ax, running_variance in zip(locations, axes.flat, running_variances):
    ax.plot(ns, running_variance)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"Running variances for {path}")
plt.show()

### plot histograms

In [ ]:
fig, axes = grid(N_locations, ncols=5)
for location, ax in zip(locations, axes.flat):
    ax.hist(data[path][:, location], bins="auto", density=True)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")
fig.suptitle(
    f"Histograms of value of {path} at {N_locations} randomly chosen locations"
)
plt.plot()

### plot qq plots

In [ ]:
fig, axes = grid(N_locations, ncols=4)
for location, ax in zip(locations, axes.flat):
    _, (slope, intercept, _) = stats.probplot(
        data[path][:, location], dist="norm", plot=ax
    )
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"QQ plots of value of {path} at {N_locations} randomly chosen locations")
plt.plot()

### Shapiro-Wilk test for normality

This is expected to fail almost always for large sample sizes.

In [ ]:
statistic, pvalue = stats.shapiro(data[path], axis=0)

alpha = 0.05


print(
    f"{(pvalue < alpha).sum()} out of {len(pvalue)} are not normal according to Shapiro-Wilk test with alpha={alpha}"
)

In [ ]:
%pip install mvshapirotest mvem

In [ ]:
from mvShapiroTest.test import mvshapiro

mvshapiro(data[path] / 100)